# COMP3710 Lab Demonstration 2 — Part 2: Eigenfaces

This notebook follows the Part 2 lab sheet and the Week 5 Demo 2 overview.

## Requirements covered
1. Load the **Labeled Faces in the Wild (LFW)** dataset.
2. Split into training and testing sets.
3. Centre the data using the training-set mean.
4. Compute PCA/eigenfaces using SVD.
5. Project train/test data into PCA **face space**.
6. Visualise eigenfaces.
7. Plot cumulative explained variance / compactness.
8. Train a **Random Forest** classifier on PCA features.
9. Report accuracy and a classification report.


In [13]:
%pip install --no-cache-dir scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

print("NumPy version:", np.__version__)


NumPy version: 2.3.0


In [16]:
import sys
print(sys.executable)

d:\PYTHON\python.exe


In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1. Load the LFW dataset

The lab uses `fetch_lfw_people(min_faces_per_person=70, resize=0.4)`.


In [2]:
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)

n_samples, h, w = lfw_people.images.shape

X = lfw_people.data
y = lfw_people.target
target_names = lfw_people.target_names

n_features = X.shape[1]
n_classes = target_names.shape[0]

print("Total dataset size:")
print("n_samples:", n_samples)
print("n_features:", n_features)
print("n_classes:", n_classes)
print("image shape:", (h, w))
print("classes:")
for i, name in enumerate(target_names):
    print(f"  {i}: {name}")


KeyboardInterrupt: 

## 2. Visualise example faces

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
axes = axes.ravel()

for i, ax in enumerate(axes):
    ax.imshow(lfw_people.images[i], cmap="gray")
    ax.set_title(target_names[y[i]], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 3. Split into training and testing sets

The lab uses a 75% / 25% split with `random_state=42`.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## 4. Centre the data

The mean is computed from the training set and subtracted from both train and test data.


In [ ]:
mean_face = np.mean(X_train, axis=0)

X_train_centered = X_train - mean_face
X_test_centered = X_test - mean_face

plt.figure(figsize=(3, 4))
plt.imshow(mean_face.reshape(h, w), cmap="gray")
plt.title("Mean Face")
plt.axis("off")
plt.show()


## 5. Compute PCA / Eigenfaces using SVD

The lab uses SVD and keeps the first 150 principal directions.


In [ ]:
n_components = 150

U, S, V = np.linalg.svd(X_train_centered, full_matrices=False)

components = V[:n_components]
eigenfaces = components.reshape((n_components, h, w))

print("U shape:", U.shape)
print("S shape:", S.shape)
print("V shape:", V.shape)
print("components shape:", components.shape)
print("eigenfaces shape:", eigenfaces.shape)


## 6. Project faces into PCA face space

In [ ]:
X_transformed = np.dot(X_train_centered, components.T)
X_test_transformed = np.dot(X_test_centered, components.T)

print("Original training shape:", X_train.shape)
print("PCA training shape:", X_transformed.shape)
print("Original testing shape:", X_test.shape)
print("PCA testing shape:", X_test_transformed.shape)


## 7. Visualise eigenfaces

In [ ]:
def plot_gallery(images, titles, h, w, n_row=3, n_col=4):
    plt.figure(figsize=(1.8 * n_col, 2.4 * n_row))
    plt.subplots_adjust(bottom=0, left=0.01, right=0.99, top=0.90, hspace=0.35)

    for i in range(n_row * n_col):
        plt.subplot(n_row, n_col, i + 1)
        plt.imshow(images[i].reshape((h, w)), cmap=plt.cm.gray)
        plt.title(titles[i], size=12)
        plt.xticks(())
        plt.yticks(())

eigenface_titles = [f"eigenface {i}" for i in range(eigenfaces.shape[0])]
plot_gallery(eigenfaces, eigenface_titles, h, w)
plt.show()


## 8. Compactness / cumulative explained variance

This follows the calculation shown in the lab sheet.


In [ ]:
explained_variance = (S ** 2) / (n_samples - 1)
total_var = explained_variance.sum()
explained_variance_ratio = explained_variance / total_var
ratio_cumsum = np.cumsum(explained_variance_ratio)

eigenvalue_count = np.arange(n_components)

plt.figure(figsize=(8, 5))
plt.plot(eigenvalue_count, ratio_cumsum[:n_components])
plt.title("Compactness")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance ratio")
plt.grid(True)
plt.show()

print(
    f"Cumulative variance captured by {n_components} components:",
    ratio_cumsum[n_components - 1]
)


## 9. Random Forest classification using PCA features

In [ ]:
estimator = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    max_features=150,
    random_state=42
)

estimator.fit(X_transformed, y_train)
predictions = estimator.predict(X_test_transformed)

correct = predictions == y_test
total_test = len(X_test_transformed)
accuracy = np.sum(correct) / total_test

print("Total Testing:", total_test)
print("Total Correct:", np.sum(correct))
print("Accuracy:", accuracy)

print()
print("Classification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=target_names
))


## 10. Show example predictions

In [ ]:
n_show = 12
fig, axes = plt.subplots(3, 4, figsize=(10, 9))
axes = axes.ravel()

for i, ax in enumerate(axes[:n_show]):
    image = X_test[i].reshape(h, w)
    true_name = target_names[y_test[i]]
    pred_name = target_names[predictions[i]]

    ax.imshow(image, cmap="gray")
    ax.set_title(f"True: {true_name}\nPred: {pred_name}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()


# 11. Demo explanation

> I loaded the Labeled Faces in the Wild dataset and split it into training and testing sets. I computed the mean face from the training data and subtracted it from both sets. Then I applied SVD to the centred training matrix. The first 150 principal directions were used as PCA components, and after reshaping them they are called eigenfaces. Each original face was projected from the high-dimensional pixel space into this lower-dimensional face space. I plotted the cumulative explained variance to evaluate how compact the PCA representation is. Finally, I used the PCA coefficients as features for a Random Forest classifier and evaluated its performance on the test set.


# 12. Likely demonstrator questions

**What is PCA doing here?**  
It finds orthogonal directions that capture the largest amount of variance in the training faces.

**What is an eigenface?**  
It is a PCA basis vector reshaped back into the dimensions of a face image.

**Why subtract the mean face first?**  
PCA is applied to centred data, so the components describe variation around the average face.

**Why use the training-set mean for the test set?**  
The test set should not influence the learned representation.

**What does `X_transformed` represent?**  
Each row is one face represented by its coordinates in the 150-dimensional PCA face space.

**What does the compactness graph show?**  
It shows the cumulative proportion of variance captured as more PCA components are included.

**Is PCA supervised?**  
No. PCA does not use class labels.

**Is Random Forest supervised?**  
Yes. It learns from PCA features together with known person labels.

**Why split into train and test sets?**  
The test set estimates performance on unseen examples.


# 13. AI-use record

Keep evidence of the prompts/questions you used, changes made after testing, errors fixed, and what each important block of code does.
